# GPU Anomaly Warning System - Colab CUDA backend
Runtime > Change runtime type > **T4 GPU** first. This notebook compiles `backend/cuda/anomaly_kernels.cu`, tests it,
benchmarks CPU vs GPU, and (optionally) serves the API so the website can use the real GPU.

Nothing here is pre-computed: every number comes from cells you run. Status of this notebook: **written but not yet executed by the authors** - report any failing cell.

## 1. Verify the GPU and compiler

In [ ]:
!nvidia-smi
!nvcc --version

## 2. Get the project
Either upload `gpu-anomaly-warning-system.zip` (option A) or clone your GitHub repo (option B).

In [ ]:
# Option A: upload the zip
from google.colab import files
up = files.upload()
!unzip -q -o gpu-anomaly-warning-system.zip -d /content
%cd /content/gpu-anomaly-warning-system

In [ ]:
# Option B: clone (edit the URL), then cd into the project
# !git clone https://github.com/<you>/gpu-anomaly-warning-system.git /content/gpu-anomaly-warning-system
# %cd /content/gpu-anomaly-warning-system

## 3. Compile the CUDA code (T4 = `sm_75`)

In [ ]:
%cd backend/cuda
!make clean && make ARCH=sm_75
!ls -la build
%cd ../..

## 4. Standalone CUDA self-test (prints PASS/FAIL, stats and timings)

In [ ]:
!./backend/cuda/build/selftest 1000000

## 5. Python tests (unit + real GPU agreement tests)

In [ ]:
!pip -q install fastapi "uvicorn[standard]" python-multipart httpx pandas numpy pytest
%cd backend
!python -m pytest tests -v -rs
%cd ..

## 6. Real CPU vs GPU benchmark
Results are written to `results/` (CSV + JSON). Download them for the Reports page / your report.

In [ ]:
%cd backend
!python -m scripts.run_benchmark --sizes 1000 10000 100000 1000000 5000000 --repeats 7
%cd ..
!ls results

In [ ]:
from google.colab import files
import glob
for f in sorted(glob.glob('results/benchmark_gpu_*.csv'))[-1:] + sorted(glob.glob('results/benchmark_gpu_*.json'))[-1:]:
    files.download(f)

## 7. (Optional) Serve the GPU API to your website
This starts the same FastAPI app with the real CUDA library and exposes it through a Cloudflare quick tunnel.
Then use **either** topology:
* **A. Frontend -> Colab directly**: paste the tunnel URL into the website *Settings > API base URL*.
* **B. Local backend -> Colab GPU**: on your machine set `GPU_MODE=remote`, `GPU_BACKEND_URL=<tunnel url>`, `GPU_BACKEND_TOKEN=<token below>`.

The tunnel URL is public while the cell runs. `GPU_API_TOKEN` protects `/api/gpu/analyze`; the other endpoints are not authenticated, so do not upload sensitive data.

In [ ]:
import os, secrets, subprocess, time, re, threading
os.environ['GPU_MODE'] = 'local'
os.environ['GPU_API_TOKEN'] = TOKEN = secrets.token_urlsafe(24)
os.environ['CORS_ORIGINS'] = '*'          # tighten to your site's origin for anything beyond a demo
os.environ['CUDA_LIB_PATH'] = '/content/gpu-anomaly-warning-system/backend/cuda/build/libanomaly.so'
server = subprocess.Popen(['python','-m','uvicorn','app.main:app','--host','127.0.0.1','--port','8000'],
                          cwd='/content/gpu-anomaly-warning-system/backend')
time.sleep(5)
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x cloudflared
tun = subprocess.Popen(['./cloudflared','tunnel','--url','http://127.0.0.1:8000'], stderr=subprocess.PIPE, text=True)
url = None
for line in tun.stderr:
    m = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', line)
    if m: url = m.group(0); break
print('GPU API URL :', url)
print('GPU API TOKEN:', TOKEN)
threading.Thread(target=lambda: [l for l in tun.stderr], daemon=True).start()

In [ ]:
# Sanity check: must say available: True and executes_cuda: True
!curl -s {url}/api/gpu/status